<a href="https://colab.research.google.com/github/futuregunmulju/2026-Signals-and-Systems-Speech-Noise-Reduction/blob/main/%5B9%EC%A1%B0%5D_Challenge1_Speech_Noise_Reduction_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [9조 구조대] 음성 잡음 제거 (고전 신호처리, 딥러닝/머신러닝 미사용)

2026 신호 및 시스템 1차 챌린지, 2차 제출 (v3).
STFT 기반 MMSE-LSA 이득 계산 1회 파이프라인에 잡음 PSD 추정(최소통계 + SPP 하이브리드), 하모닉 콤, 사이렌 궤적 추적, 톤 억제를 연결.

In [ ]:
%pip install -q "numpy>=1.26,<3" "scipy>=1.12,<2" "soundfile>=0.12,<1"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


In [ ]:
from pathlib import Path

import numpy as np
import soundfile as sf
from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.special import i0, i1, exp1


INPUT_DIR = Path("input")
OUTPUT_DIR = Path("output")

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"입력 폴더: {INPUT_DIR.resolve()}")
print(f"출력 폴더: {OUTPUT_DIR.resolve()}")
print("input 폴더에 테스트 WAV를 넣은 뒤 나머지 셀을 실행하세요.")

## 알고리즘

1. 고역통과(100 Hz, 영위상) → 2. 정상 톤 억제 → 3. 잡음 PSD 추정: 비인과 최소통계 + 양방향 SPP(음성 존재 확률) 추적을 로그 영역에서 혼합
4. 사이렌 궤적 추적(검출 시에만 잡음 PSD에 반영) → 5. Decision-directed + 케프스트럼 평활 사전 SNR
6. 하모닉 콤 → 7. MMSE-LSA 이득 + 휴지 구간 이득 하한 → 8. 이득 평활

v3에서 바뀐 점: 3단계에 SPP 기반 잡음 추적을 추가하고, 잡음 수준 관련 파라미터를 포함해 전체를 재튜닝했습니다(자세한 내용은 README).
튜닝된 파라미터는 코드 안에 내장되어 있어 외부 파일이 필요 없습니다.

In [ ]:
"""Cascade speech denoiser (classical DSP only, no ML).

Stage 0 : zero-phase high-pass (wind / rumble)
Stage 1 : sweeping-tone (siren/alarm) suppression, optional
Stage 2 : two-pass NON-CAUSAL noise PSD estimation over the whole clip
Stage 3 : decision-directed MMSE-LSA gain
Stage 4 : TSNR refinement (Plapous et al., 2006)
Stage 5 : musical-noise suppression (gain smoothing in low-gain regions)
Stage 6 : optional leak of the high-passed noisy signal

"""
import json
import os

import numpy as np
import soundfile as sf
from scipy.ndimage import gaussian_filter1d, minimum_filter1d, uniform_filter, uniform_filter1d
from scipy.ndimage import maximum_filter1d, median_filter as _medfilt
from scipy.signal import butter, istft, sosfiltfilt, stft
from scipy.special import exp1

DEFAULTS = dict(
    hp_hz=60.0,
    n_fft=512, hop=128,
    min_win=120,
    min_bias=1.7,
    delta=3.0,
    noise_sigma=40.0,
    noise_scale=1.1,
    n_iter=2,
    alpha_dd=0.96,
    gain_floor_db=-18.0,
    tsnr=True,
    gain_smooth=True,
    leak=0.0,
    tonal_db=0.0,          # 0 = off
    tonal_max_cut_db=20.0,
    tonal_max_peaks=5,
    ceps_w=0.0,
    ceps_win=9,
    ceps_q_lo_ms=1.5,
    floor_pause_db=None,
    pause_thr=2.0,
    siren_w=0.0,
    siren_lo=500.0, siren_hi=2000.0, siren_fmax=7000.0,
    siren_med=41,
    siren_thr=6.0,
    siren_pers=0.6,
    siren_J=6, siren_lam=0.5,
    siren_bw=2,
    h_w=0.0,
    h_lo=0.4,
    h_pow=3.0,
    h_z0=2.5,
    h_fmax=2500.0,
    spp_w=0.0, spp_sigma=15.0, spp_xi_db=15.0, spp_alpha=0.8,
    spp_bias=1.0, spp_ns0=0.0, spp_ns_scale=1.0,
)

# 튜닝 결과(Train 탐색 + Validation 확인)를 파일 없이 코드에 직접 내장
DEFAULTS.update({
    'hp_hz': 100.0,
    'n_fft': 640,
    'hop': 160,
    'min_win': 120,
    'min_bias': 2.1590776426974423,
    'delta': 3.1082132332750425,
    'noise_sigma': 48.26717981051946,
    'noise_scale': 1.589896186320649,
    'n_iter': 1,
    'alpha_dd': 0.9494649511749627,
    'gain_floor_db': -24.0,
    'tsnr': False,
    'gain_smooth': True,
    'leak': 0.008613377713640836,
    'tonal_db': 18.0,
    'tonal_max_cut_db': 20.0,
    'tonal_max_peaks': 5,
    'ceps_w': 0.1295990600247068,
    'ceps_win': 25,
    'ceps_q_lo_ms': 1.2307957741480489,
    'floor_pause_db': -28.091237542032918,
    'pause_thr': 1.0592393394348312,
    'h_w': 0.376253493108925,
    'h_lo': 0.3532402441107253,
    'h_pow': 2.0,
    'h_z0': 2.2363079034832873,
    'h_fmax': 2500.0,
    'siren_w': 0.9400149909373927,
    'siren_lo': 500.0,
    'siren_hi': 2000.0,
    'siren_fmax': 7000.0,
    'siren_med': 41,
    'siren_thr': 6.0,
    'siren_pers': 0.4915748635476811,
    'siren_J': 10,
    'siren_lam': 0.5,
    'siren_bw': 2,
    'spp_w': 0.3068935374498033,
    'spp_sigma': 2.0,
    'spp_xi_db': 14.62247855028253,
    'spp_alpha': 0.8336048442600156,
    'spp_bias': 1.6,
    'spp_ns0': 0.10145702026474249,
    'spp_ns_scale': 0.6711397164760309,
})


def _noise_psd_base(P, p):
    S = uniform_filter1d(P, 3, axis=0, mode="nearest")
    S = uniform_filter1d(S, 5, axis=1, mode="nearest")
    win = int(min(p["min_win"], P.shape[1]))
    n_min = minimum_filter1d(S, size=win, axis=1, mode="nearest") * p["min_bias"]
    n_min = np.maximum(gaussian_filter1d(n_min, 5, axis=1, mode="nearest"), 1e-14)

    noise = n_min.copy()
    for _ in range(int(p["n_iter"])):
        speech = (S > p["delta"] * noise).astype(float)
        presence = uniform_filter(speech, size=(3, 5), mode="nearest")
        w = (1.0 - presence) ** 2
        num = gaussian_filter1d(w * P, p["noise_sigma"], axis=1, mode="nearest")
        den = gaussian_filter1d(w, p["noise_sigma"], axis=1, mode="nearest")
        lam = np.clip(den / 0.3, 0.0, 1.0)
        avg = num / np.maximum(den, 1e-9)
        noise = np.maximum(lam * avg + (1 - lam) * n_min, 1e-14)
    return noise * p["noise_scale"]


def _spp_pass(P, N0, xi_db, alpha, bias):
    """Gerkmann-Hendriks 계열 SPP 기반 잡음 파워 추적(한 방향)."""
    xi = 10 ** (xi_db / 10)
    k = xi / (1 + xi)
    sig = N0[:, 0].copy()
    pbar = np.full(P.shape[0], 0.5)
    out = np.empty_like(P)
    for t in range(P.shape[1]):
        post = 1.0 / (1.0 + (1 + xi) * np.exp(-(P[:, t] / np.maximum(sig, 1e-14)) * k))  # P(음성 존재|y)
        pbar = 0.9 * pbar + 0.1 * post
        post = np.where(pbar > 0.99, np.minimum(post, 0.99), post)                       # 정체 방지
        en = (1 - post) * P[:, t] + post * sig
        sig = np.maximum(alpha * sig + (1 - alpha) * en, 1e-14)
        out[:, t] = sig
    return out * bias


def _noise_psd(P, p):
    """기존 최소통계 추정(N0)과 양방향 SPP 추정을 로그 영역에서 혼합."""
    N0 = _noise_psd_base(P, p)
    w0 = p["spp_w"]
    if w0 <= 0:
        return N0
    a, xdb, b = p["spp_alpha"], p["spp_xi_db"], p["spp_bias"]
    f = _spp_pass(P, N0, xdb, a, b)
    r = _spp_pass(P[:, ::-1], N0[:, ::-1], xdb, a, b)[:, ::-1]
    Ns = gaussian_filter1d(np.exp(0.5 * (np.log(f) + np.log(r))), p["spp_sigma"], axis=1, mode="nearest")
    K = P.shape[0]
    e = np.linspace(0, K, 9).astype(int)
    L = 10 * np.log10(np.array([Ns[e[i]:e[i + 1]].mean(axis=0) for i in range(8)]) + 1e-14)
    ns = float(L.std(axis=1).mean())            # 입력에서 직접 잰 시간 변동 폭(dB)
    gate = min(max((ns - p["spp_ns0"]) / max(p["spp_ns_scale"], 1e-3), 0.0), 1.0)
    w = w0 * gate
    return np.exp((1 - w) * np.log(N0) + w * np.log(np.maximum(Ns, 1e-14)))


def _tonal_suppress(x, sr, thr_db, max_cut_db, max_peaks):
    """Sparse, prominent, time-continuous spectral peaks (siren/alarm, also sweeping)
    are notched out. Voiced speech has many harmonic peaks per frame -> protected."""
    n_fft, hop = 1024, 128
    _, _, Y = stft(x, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop)
    mag_db = 20 * np.log10(np.abs(Y) + 1e-9)
    K, T = mag_db.shape
    pad = np.pad(mag_db, ((24, 24), (0, 0)), mode="edge")
    offs = [*range(-24, -5), *range(6, 25)]
    side = np.median(np.stack([pad[24 + o:24 + o + K] for o in offs]), axis=0)
    prom = mag_db - side
    peak = (mag_db >= np.roll(mag_db, 1, 0)) & (mag_db >= np.roll(mag_db, -1, 0)) & (prom >= thr_db)
    peak[:4] = False
    sparse = peak.sum(axis=0) <= max_peaks
    peak &= sparse[None, :]
    near = uniform_filter(peak.astype(float), size=(7, 1), mode="constant") > 0
    persistent = uniform_filter1d(near.astype(float), size=5, axis=1, mode="constant") >= 0.6
    cand = peak & persistent
    cut = np.where(cand, np.minimum(prom - thr_db + 6.0, max_cut_db), 0.0)
    cut = uniform_filter(cut, size=(5, 1), mode="constant") * 1.5
    cut = np.minimum(gaussian_filter1d(cut, 1.0, axis=1), max_cut_db)
    _, y = istft(Y * 10 ** (-cut / 20), fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop)
    y = y[:len(x)]
    return np.pad(y, (0, len(x) - len(y))) if len(y) < len(x) else y


def _ceps_xi(gamma, n_fft, sr, win, q_lo_ms, xi_min):
    """Cepstral smoothing of the ML SNR estimate (Breithaupt et al. 2007/2008, non-causal version)."""
    lg = np.log(np.maximum(gamma, 1e-6))
    c = np.fft.irfft(lg, n=n_fft, axis=0)
    n = n_fft
    q_lo = max(int(q_lo_ms * 1e-3 * sr), 2)
    keep = np.zeros_like(c, dtype=bool)
    keep[:q_lo + 1] = True
    keep[n - q_lo:] = True
    qa, qb = int(sr / 400), int(sr / 80)
    qb = min(qb, n // 2 - 3)
    qp = qa + np.argmax(c[qa:qb + 1], axis=0)
    T = c.shape[1]
    idx = np.arange(T)
    for d in range(-2, 3):
        for mult in (1, 2):
            q = np.clip(mult * qp + d, 0, n // 2)
            keep[q, idx] = True
            keep[n - q, idx] = True
    c_s = np.where(keep, c, uniform_filter1d(c, int(win), axis=1, mode="nearest"))
    lg_s = np.fft.rfft(c_s, axis=0).real
    gs = np.exp(lg_s + 0.5772156649)
    return np.maximum(gs - 1.0, xi_min)


def _siren_track(P, fr, p):
    """Track a (sweeping) harmonic siren/alarm: spectral-peak prominence -> harmonic-comb salience over a
    fundamental grid -> Viterbi path with a bounded sweep rate.  Returns (S, info); S = siren power estimate."""
    K, T = P.shape
    df = fr[1] - fr[0]
    base = _medfilt(P, size=(int(p["siren_med"]), 1), mode="nearest")
    prom = np.minimum(np.maximum(10 * np.log10(P / np.maximum(base, 1e-14)), 0.0), 25.0)
    prom = maximum_filter1d(prom, 3, axis=0)
    f0s = np.arange(p["siren_lo"], p["siren_hi"], df / 2.0)
    nC = len(f0s)
    num = np.zeros((nC, T)); den = np.zeros(nC)
    for h in range(1, int(p["siren_fmax"] / p["siren_lo"]) + 1):
        kb = np.rint(h * f0s / df).astype(int)
        km = np.rint((h + 0.5) * f0s / df).astype(int)
        ok = (km < K - 1) & ((h + 0.5) * f0s <= p["siren_fmax"])
        w = 1.0 / np.sqrt(h)
        num[ok] += w * (prom[kb[ok]] - prom[km[ok]])
        den[ok] += w
    sal = num / np.maximum(den, 1e-9)[:, None]
    J = int(p["siren_J"]); lam = p["siren_lam"]
    dp = np.zeros((nC, T)); bp = np.zeros((nC, T), dtype=int)
    dp[:, 0] = sal[:, 0]
    shifts = list(range(-J, J + 1))
    for t in range(1, T):
        cand = np.full((len(shifts), nC), -1e9)
        for si, d in enumerate(shifts):
            if d >= 0:
                cand[si, d:] = dp[:nC - d, t - 1] - lam * abs(d) if d else dp[:, t - 1]
            else:
                cand[si, :nC + d] = dp[-d:, t - 1] - lam * abs(d)
        best = np.argmax(cand, axis=0)
        dp[:, t] = sal[:, t] + cand[best, np.arange(nC)]
        bp[:, t] = np.arange(nC) - np.array(shifts)[best]
    path = np.zeros(T, dtype=int)
    path[-1] = int(np.argmax(dp[:, -1]))
    for t in range(T - 1, 0, -1):
        path[t - 1] = bp[path[t], t]
    ps = sal[path, np.arange(T)]
    on = uniform_filter1d((ps > p["siren_thr"]).astype(float), 5, mode="nearest") > 0.5
    info = dict(frac_on=float(on.mean()), ps_med=float(np.median(ps)), f0=f0s[path])
    S = np.zeros_like(P)
    if info["frac_on"] >= p["siren_pers"]:
        f0t = uniform_filter1d(f0s[path], 3, mode="nearest")
        hop_s = float(p["hop"]) / 16000.0
        rate = np.abs(np.gradient(f0t)) / hop_s
        twin = float(p["n_fft"]) / 16000.0
        for t in np.where(on)[0]:
            for h in range(1, int(p["siren_fmax"] / p["siren_lo"]) + 1):
                fc = h * f0t[t]
                if fc > p["siren_fmax"] or fc / df >= K - 1:
                    break
                half = (p["siren_bw"] * df + 0.5 * h * rate[t] * twin) / df
                lo_, hi_ = max(int(np.floor(fc / df - half)), 0), min(int(np.ceil(fc / df + half)) + 1, K)
                S[lo_:hi_, t] = np.maximum(P[lo_:hi_, t] - base[lo_:hi_, t], 0.0)
    return S, info


def siren_flag(x, sr=16000, **overrides):
    """진단용: 이 클립에서 사이렌이 검출되는가 -> (검출 여부, frac_on, ps_med)"""
    p = {**DEFAULTS, **overrides}
    x = np.asarray(x, dtype=np.float64)
    xh = sosfiltfilt(butter(4, p["hp_hz"], btype="highpass", fs=sr, output="sos"), x) if p["hp_hz"] > 0 else x
    n_fft, hop = int(p["n_fft"]), int(p["hop"])
    fr, _, Y = stft(xh, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop)
    S, info = _siren_track(np.abs(Y) ** 2 + 1e-14, fr, p)
    return info["frac_on"] >= p["siren_pers"], info["frac_on"], info["ps_med"]


def _harmonic_mult(P, fr, sr, n_fft, h_lo, h_pow, z0, fmax):
    """Pitch-based harmonic comb (cepstral F0 per frame). m(K,T) in [h_lo,1], voicing confidence v(T)."""
    c = np.fft.irfft(np.log(P), n=n_fft, axis=0)
    qa, qb = int(sr / 400), int(sr / 80)
    qb = min(qb, n_fft // 2 - 3)
    seg = c[qa:qb + 1]
    seg = seg - uniform_filter1d(seg, 15, axis=0, mode="nearest")
    Q, T = seg.shape
    idx = np.arange(T)
    i = np.argmax(seg, axis=0)
    peak = seg[i, idx]
    z = (peak - seg.mean(axis=0)) / (seg.std(axis=0) + 1e-9)
    y0, y2 = seg[np.clip(i - 1, 0, Q - 1), idx], seg[np.clip(i + 1, 0, Q - 1), idx]
    den = y0 - 2 * peak + y2
    delta = np.clip(np.where(np.abs(den) > 1e-12, 0.5 * (y0 - y2) / den, 0.0), -1, 1)
    q = _medfilt(qa + i + delta, size=5, mode="nearest")
    f0 = sr / q
    v = uniform_filter1d(np.clip((z - z0) / 2.0, 0.0, 1.0), 3, mode="nearest")
    comb = (0.5 * (1 + np.cos(2 * np.pi * fr[:, None] / f0[None, :]))) ** h_pow
    m = h_lo + (1 - h_lo) * comb
    m = np.where(fr[:, None] <= fmax, m, 1.0)
    return m, v


def _lsa(xi, gamma):
    r = xi / (1.0 + xi)
    v = np.maximum(r * gamma, 1e-8)
    return r * np.exp(0.5 * exp1(v))


def denoise_channel(x, sr=16000, **overrides):
    p = {**DEFAULTS, **overrides}
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    if n < p["n_fft"] or not np.any(x):
        return x.copy()

    if p["hp_hz"] > 0:
        sos = butter(4, p["hp_hz"], btype="highpass", fs=sr, output="sos")
        xh = sosfiltfilt(sos, x)
    else:
        xh = x

    if p["tonal_db"] > 0:
        xh = _tonal_suppress(xh, sr, p["tonal_db"], p["tonal_max_cut_db"], int(p["tonal_max_peaks"]))

    n_fft, hop = int(p["n_fft"]), int(p["hop"])
    fr, _, Y = stft(xh, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop)
    P = np.abs(Y) ** 2 + 1e-14
    N = _noise_psd(P, p)
    if p["siren_w"] > 0:
        S_, _ = _siren_track(P, fr, p)
        N = N + p["siren_w"] * S_

    floor = 10 ** (p["gain_floor_db"] / 20)
    K, T = P.shape
    xi_min = 10 ** (-25 / 10)
    XC = None
    if p["ceps_w"] > 0:
        XC = _ceps_xi(np.minimum(P / N, 1000.0), n_fft, sr, p["ceps_win"], p["ceps_q_lo_ms"], xi_min)
    HM = None
    if p["h_w"] > 0:
        m_, v_ = _harmonic_mult(P, fr, sr, n_fft, p["h_lo"], p["h_pow"], p["h_z0"], p["h_fmax"])
        HM = 1.0 - p["h_w"] * v_[None, :] * (1.0 - m_)
    fp_db = p["floor_pause_db"]
    if fp_db is None:
        floor_t = np.full(T, floor)
    else:
        band = (fr >= 200) & (fr <= 3500)
        m = np.minimum(P / N, 30.0)[band].mean(axis=0)
        pres = np.clip((m - 1.0) / max(p["pause_thr"], 1e-3), 0.0, 1.0)
        pres = uniform_filter1d(pres, 7, mode="nearest")
        floor_t = 10 ** ((p["gain_floor_db"] * pres + fp_db * (1 - pres)) / 20)
    G = np.empty_like(P)
    prev = np.zeros(K)
    for t in range(T):
        gamma = np.minimum(P[:, t] / N[:, t], 1000.0)
        xi = np.maximum(p["alpha_dd"] * prev + (1 - p["alpha_dd"]) * np.maximum(gamma - 1, 0), xi_min)
        if XC is not None:
            w = p["ceps_w"]
            xi = np.exp((1 - w) * np.log(xi) + w * np.log(XC[:, t]))
        if HM is not None:
            xi = np.maximum(xi * HM[:, t], xi_min)
        g = np.clip(_lsa(xi, gamma), floor_t[t], 1.0)
        G[:, t] = g
        prev = g ** 2 * gamma

    if p["tsnr"]:
        xi2 = G ** 2 * P / N
        G = np.clip(xi2 / (1.0 + xi2), floor_t[None, :], 1.0)

    if p["gain_smooth"]:
        Gs = uniform_filter(G, size=(3, 3), mode="nearest")
        G = np.where(G < 0.4, Gs, G)

    _, y = istft(Y * G, fs=sr, window="hann", nperseg=n_fft, noverlap=n_fft - hop)
    y = y[:n]
    if len(y) < n:
        y = np.pad(y, (0, n - len(y)))
    if p["leak"] > 0:
        y = y + p["leak"] * xh
    return np.nan_to_num(y)


def denoise(x, sr=16000, **overrides):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 1:
        return denoise_channel(x, sr, **overrides)
    return np.stack([denoise_channel(x[:, c], sr, **overrides) for c in range(x.shape[1])], axis=1)

SR = 16000  # 입력 검증 셀에서 사용


## 실행

`input/` 폴더에 테스트 WAV를 넣고 아래 셀을 실행하면 같은 파일명으로 `output/`에 저장됩니다.

In [ ]:
wav_files = sorted(
    path
    for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() == ".wav"
)

if not wav_files:
    raise FileNotFoundError(f"테스트 WAV를 먼저 넣어주세요: {INPUT_DIR.resolve()}")

failures = []
success_count = 0

for index, input_path in enumerate(wav_files, start=1):
    output_path = OUTPUT_DIR / input_path.name
    temporary_path = OUTPUT_DIR / f".{input_path.name}.tmp"

    try:
        noisy, sample_rate = sf.read(input_path, dtype="float64", always_2d=True)

        if sample_rate != SR:
            raise ValueError(f"16 kHz 입력이 필요합니다. 현재 {sample_rate} Hz")
        if len(noisy) == 0 or not np.isfinite(noisy).all():
            raise ValueError("입력 음성이 비어 있거나 유효하지 않습니다.")

        enhanced = np.empty_like(noisy)
        for channel in range(noisy.shape[1]):
            enhanced[:, channel] = denoise(noisy[:, channel])

        if enhanced.shape != noisy.shape:
            raise ValueError("처리 후 샘플 수 또는 채널 수가 달라졌습니다.")
        if not np.isfinite(enhanced).all():
            raise ValueError("출력에 유효하지 않은 값이 있습니다.")

        sf.write(temporary_path, enhanced, sample_rate, format="WAV", subtype="FLOAT")

        info = sf.info(temporary_path)
        if (
            info.samplerate != sample_rate
            or info.channels != noisy.shape[1]
            or info.frames != noisy.shape[0]
        ):
            raise ValueError("저장된 WAV의 형식이 입력과 일치하지 않습니다.")

        temporary_path.replace(output_path)
        success_count += 1
        print(f"[{index}/{len(wav_files)}] 저장 완료: {output_path.name}")

    except Exception as error:
        temporary_path.unlink(missing_ok=True)
        failures.append((input_path.name, str(error)))
        print(f"[{index}/{len(wav_files)}] 실패: {input_path.name} — {error}")

print(f"\n처리 완료: {success_count}/{len(wav_files)}개")
print(f"출력 폴더: {OUTPUT_DIR.resolve()}")

if failures:
    raise RuntimeError(f"{len(failures)}개 파일 처리에 실패했습니다. 위 오류를 확인하세요.")

## Validation 결과 (PESQ-WB, 164쌍)

평균 **1.8609** (SNR 0/5/10/15 dB: 1.264 / 1.593 / 1.876 / 2.188)
이전 버전 v2: 1.8395 (1.253 / 1.571 / 1.865 / 2.154)